# Galaxy Defender — Android APK (Google Colab)

On your Redmi Pad 2:

1. Sign in with Google if asked.
2. Tap **Runtime → Run all** (or **Runtime → Run all cells**).
3. Wait **25–50 minutes**. Keep this tab open and the pad plugged in.
4. When the last cell finishes, it downloads `galaxydefender-1.0-arm64-v8a-debug.apk`.
5. Open that file → **Install**. Android may ask you to allow installs from Chrome/Files.

This is a **debug** APK (fine to play at home). Google Colab does the compiling — I cannot press Run on your Google account from here.

If it fails, you can still play Galaxy Defender in the Grok preview.

In [ ]:
from IPython.display import Javascript, display
display(Javascript('''
function KeepAlive(){
  console.log("colab keep-alive");
  const b = document.querySelector("colab-connect-button") || document.querySelector("#connect");
  if (b) b.click();
}
setInterval(KeepAlive, 60000);
'''))

import os, sys, subprocess, shutil
print("Notebook Python", sys.version)

def sh(cmd):
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    subprocess.check_call(cmd, shell=isinstance(cmd, str))

sh("sudo apt-get update -qq")
sh("sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
   "python3.11 python3.11-venv python3.11-dev python3.11-distutils "
   "build-essential git zip unzip autoconf libtool pkg-config "
   "openjdk-17-jdk zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo5 "
   "cmake libffi-dev libssl-dev dos2unix unzip wget || true")

# distutils package name varies; ignore if missing
sh("sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq python3.11 python3.11-venv python3.11-dev openjdk-17-jdk")

java_home = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = java_home + "/bin:" + os.environ.get("PATH", "")
sh(["java", "-version"])

venv = "/content/bz"
if not os.path.isfile(venv + "/bin/python"):
    sh(["python3.11", "-m", "venv", venv])
pip = venv + "/bin/pip"
py = venv + "/bin/python"
sh([pip, "install", "-q", "-U", "pip", "wheel"])
sh([pip, "install", "-q", "Cython==0.29.36", "setuptools<71.0.0", "buildozer", "virtualenv", "sh", "pexpect"])
os.environ["PATH"] = venv + "/bin:" + os.environ["PATH"]
sh(["buildozer", "--version"])
print("Setup done")

In [ ]:
import os, glob, shutil, subprocess, textwrap

ROOT = "/content/Galaxy-Defender"
if os.path.isdir(ROOT):
    shutil.rmtree(ROOT)

subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/AmarGams/Galaxy-Defender.git", ROOT])
os.chdir(ROOT)
print("cwd", os.getcwd(), "files", os.listdir(".")[:20])
assert os.path.isfile("main.py"), "main.py missing — clone failed"

spec = textwrap.dedent('''\
[app]

title = Galaxy Defender
package.name = galaxydefender
package.domain = org.amargams

source.dir = .
source.include_exts = py,png,jpg,jpeg,gif,mp3,wav,ogg,txt
source.exclude_dirs = tests, bin, .github, .git, .buildozer
source.exclude_patterns = Galaxy-Defender-Pydroid3.zip,*.yml,*.yaml,*.md,*.zip,*.ipynb

version = 1.0

requirements = python3==3.11.10,hostpython3==3.11.10,pygame
orientation = landscape
fullscreen = 1

android.permissions = INTERNET,VIBRATE
android.api = 33
android.minapi = 24
android.accept_sdk_license = True
android.archs = arm64-v8a
p4a.branch = master
p4a.bootstrap = sdl2

[buildozer]
log_level = 2
warn_on_root = 0
''')
open("buildozer.spec", "w").write(spec)
print("Wrote buildozer.spec")

# Audio must not crash the APK if the mixer fails on a device.
src = open("main.py").read()
old = '''pygame.init()\npygame.mixer.init()\n\npygame.mixer.music.load("music.mp3")\npygame.mixer.music.play(-1)'''
new = '''pygame.init()\ntry:\n    pygame.mixer.init()\n    pygame.mixer.music.load("music.mp3")\n    pygame.mixer.music.play(-1)\nexcept Exception:\n    pass'''
if old in src:
    open("main.py", "w").write(src.replace(old, new, 1))
    print("Patched mixer so a silent device still launches")
else:
    print("Mixer already patched or source changed — leaving main.py as cloned")

env = os.environ.copy()
env["PATH"] = "/content/bz/bin:/usr/lib/jvm/java-17-openjdk-amd64/bin:" + env.get("PATH", "")
env["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
print("Starting buildozer (25–50 min first time)…")
r = subprocess.run(["buildozer", "-v", "android", "debug"], env=env)
apks = glob.glob("bin/*.apk")
print("exit", r.returncode, "APKs", apks)
if r.returncode != 0 or not apks:
    raise SystemExit("Build failed. Scroll up for # command failed / Error compiling. This method can still fail — play in the Grok preview if it does.")

In [ ]:
import glob, os
from google.colab import files

apks = sorted(glob.glob("/content/Galaxy-Defender/bin/*.apk"))
print("Ready:", apks)
if not apks:
    raise SystemExit("No APK to download. The build cell did not succeed.")
for apk in apks:
    print(apk, round(os.path.getsize(apk) / 1024 / 1024, 1), "MB")
    files.download(apk)